In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
REPOSITORY_ROOT = PROJECT_ROOT / "sample_repository"

print(REPOSITORY_ROOT)
print(REPOSITORY_ROOT.exists())

/Users/theo/Documents/Projects/programingProjects/agentic_AI_project/agentic-ai/sample_repository
True


In [3]:
model = OpenAIChatModel(
    model_name="qwen3:4b-instruct",
    provider=OllamaProvider(
        base_url="http://localhost:11434/v1"
    ),
)
agent = Agent(model)

---

## creating a filesystem tool

In [4]:
@agent.tool_plain
def list_repository_files() -> list[str]:
    """Return the relative paths of files in the repository."""
    return [
        str(path.relative_to(REPOSITORY_ROOT))
        for path in REPOSITORY_ROOT.rglob("*")
        if path.is_file()
    ]
    
    

@agent.tool_plain
def read_repository_file(path: str) -> str:
    """Read a text file from the repository."""
    file_path = (REPOSITORY_ROOT / path).resolve()

    if not file_path.is_relative_to(REPOSITORY_ROOT.resolve()):
        raise ValueError("Path is outside the repository.")

    if not file_path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    return file_path.read_text(encoding="utf-8")

In [6]:
files = list_repository_files()
print(files)
print(read_repository_file("user_service.py"))

['test_user_service.py', 'app.py', 'user_service.py']
USERS = {
    "1": "Theo",
    "2": "Alice",
    "3": "Bob",
}


def get_user_name(user_id: str) -> str:
    return USERS[user_id]


---

---

---

## making the agent use do something

In [9]:
result = await agent.run(
    """
    Inspect the repository.

    First determine which files exist.
    Then inspect the files relevant to the user service.

    Explain:
    1. Which files are relevant.
    2. What the user service does.
    3. One potential problem in the implem*entation.

    You must use the available repository tools.
    """
)

print(result.output)

### 1. Relevant Files
The files relevant to the user service are:
- `user_service.py`: Contains the core logic for retrieving user names by ID.
- `app.py`: Contains the main application logic that interacts with the user service.

### 2. What the User Service Does
The user service (`user_service.py`) provides a function `get_user_name(user_id: str)` that retrieves a user's name based on their ID. It uses a dictionary (`USERS`) to store user IDs and their corresponding names. For example:
- User ID "1" maps to "Theo"
- User ID "2" maps to "Alice"
- User ID "3" maps to "Bob"

The application (`app.py`) runs a simple interactive loop where it prompts the user to enter a user ID, calls `get_user_name()` from the user service, and prints the corresponding user name.

### 3. One Potential Problem in the Implementation
A critical issue is **error handling**. The `get_user_name` function does not handle invalid user IDs. If a user enters a user ID that does not exist in the `USERS` dictionary 

In [10]:
for message in result.new_messages():
    print(message)
    print("-" * 80)

ModelRequest(parts=[UserPromptPart(content='\n    Inspect the repository.\n\n    First determine which files exist.\n    Then inspect the files relevant to the user service.\n\n    Explain:\n    1. Which files are relevant.\n    2. What the user service does.\n    3. One potential problem in the implem*entation.\n\n    You must use the available repository tools.\n    ', timestamp=datetime.datetime(2026, 9, 18, 12, 41, 3, 519857, tzinfo=datetime.timezone.utc))])
--------------------------------------------------------------------------------
ModelResponse(parts=[ToolCallPart(tool_name='list_repository_files', args='{}', tool_call_id='xrnsgz570PhR3rryzUAS2IRVDiJiZCAU')], usage=RequestUsage(input_tokens=266, output_tokens=17), model_name='qwen3:4b-instruct', timestamp=datetime.datetime(2026, 9, 18, 12, 41, 4, tzinfo=TzInfo(0)), provider_name='ollama', provider_response_id='chatcmpl-389')
--------------------------------------------------------------------------------
ModelRequest(parts=[

There are now two different levels of execution.
Your Python program
Your notebook controls:
```
Agent
Tools
Ollama connection
Repository access
```

The model
The model decides:
```
Which tool should I use?
What arguments should I provide?
Do I need another tool?
When should I stop?
```
That distinction is the central idea we have been building toward.